# 01. Most Popular Repo 기준 탐색

"인기 있는 repo"의 기준은 하나가 아닙니다.

여기서는 GitHub Archive 이벤트 데이터를 기반으로 여러 기준을 만들어보고, 랭킹이 어떻게 달라지는지 비교합니다.

## 1. 데이터 로드

In [1]:
from pathlib import Path

import pandas as pd

from gharchive.transform import optimize_types

OUTPUT_DIR = Path("../../data/daily_agg")

df = pd.read_parquet(OUTPUT_DIR)
df = optimize_types(df)

print(f"Rows: {len(df):,}")
print(f"Event types: {sorted(df['type'].unique())}")

Rows: 38,820,324


Event types: ['CommitCommentEvent', 'CreateEvent', 'DeleteEvent', 'DiscussionEvent', 'ForkEvent', 'GollumEvent', 'IssueCommentEvent', 'IssuesEvent', 'MemberEvent', 'PublicEvent', 'PullRequestEvent', 'PullRequestReviewCommentEvent', 'PullRequestReviewEvent', 'PushEvent', 'ReleaseEvent', 'WatchEvent']


## 2. Event Type별 의미

| Event Type | 의미 | Popularity 관련성 |
|---|---|---|
| **WatchEvent** | Star를 누른 것 | ⭐ 직접적 인기 지표 |
| **ForkEvent** | Fork한 것 | 활용/참여 의사 |
| **PushEvent** | 코드 Push | 개발 활발도 |
| **IssuesEvent** | Issue 생성/수정 | 커뮤니티 활동 |
| **PullRequestEvent** | PR 생성/수정 | 기여 활동 |
| **CreateEvent** | Branch/Tag 생성 | 개발 활동 |
| **IssueCommentEvent** | Issue 댓글 | 토론 활발도 |
| **DeleteEvent** | Branch/Tag 삭제 | 정리 활동 |
| **ReleaseEvent** | 릴리스 발행 | 프로젝트 성숙도 |

## 3. Repo별 Event Type 집계 (Pivot)

In [2]:
repo_events = (
    df.groupby(["repo_id", "type"], observed=True)["cnt"]
    .sum()
    .reset_index()
    .pivot_table(index="repo_id", columns="type", values="cnt", fill_value=0)
)

print(f"Repos: {len(repo_events):,}")
print(f"Columns: {list(repo_events.columns)}")
repo_events.head()

Repos: 10,369,896
Columns: ['CommitCommentEvent', 'CreateEvent', 'DeleteEvent', 'DiscussionEvent', 'ForkEvent', 'GollumEvent', 'IssueCommentEvent', 'IssuesEvent', 'MemberEvent', 'PublicEvent', 'PullRequestEvent', 'PullRequestReviewCommentEvent', 'PullRequestReviewEvent', 'PushEvent', 'ReleaseEvent', 'WatchEvent']


type,CommitCommentEvent,CreateEvent,DeleteEvent,DiscussionEvent,ForkEvent,GollumEvent,IssueCommentEvent,IssuesEvent,MemberEvent,PublicEvent,PullRequestEvent,PullRequestReviewCommentEvent,PullRequestReviewEvent,PushEvent,ReleaseEvent,WatchEvent
repo_id,,,,,,,,,,,,,,,,
1,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,3.0
27,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
130,0.0,35.0,23.0,0.0,0.0,0.0,6.0,30.0,0.0,0.0,50.0,99.0,111.0,189.0,0.0,0.0
144,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
230,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## 4. 기준 1: WatchEvent(Star) 총합 Top-N

In [3]:
TOP_N = 30

if "WatchEvent" in repo_events.columns:
    star_ranking = repo_events["WatchEvent"].nlargest(TOP_N)
    print(f"Top {TOP_N} repos by WatchEvent (stars):")
    print(star_ranking.to_frame("stars"))
else:
    print("WatchEvent column not found")

Top 30 repos by WatchEvent (stars):
              stars
repo_id            
1103012935  32056.0
1075372545   8670.0
997737944    8396.0
1130564872   8183.0
1073224795   7959.0
1136590548   7706.0
1156956890   7673.0
1174820787   7397.0
1061953414   5998.0
1153093670   5938.0
1141704318   5662.0
1170821064   5218.0
1149707767   5143.0
872119017    4956.0
975734319    4685.0
943398999    4608.0
1171026502   4478.0
1104332987   4139.0
1146738089   4118.0
1158588224   4064.0
896924279    4047.0
1116260703   4022.0
1134426800   4017.0
1147094660   3947.0
1166139870   3919.0
1141782198   3369.0
1031059905   3337.0
1087192965   3159.0
54346799     3108.0
1035029907   3089.0


## 5. 기준 2: 가중 점수

이벤트 종류별 가중치를 부여해서 종합 점수를 계산합니다.

가중치는 자유롭게 조절 가능합니다.

In [4]:
WEIGHTS: dict[str, float] = {
    "WatchEvent": 1.0,
    "ForkEvent": 2.0,
    "IssuesEvent": 0.5,
    "PullRequestEvent": 3.0,
    "IssueCommentEvent": 0.3,
    "PushEvent": 0.2,
}


def compute_weighted_score(
    repo_events: pd.DataFrame, weights: dict[str, float]
) -> pd.Series:
    """Compute weighted popularity score per repo."""
    score = pd.Series(0.0, index=repo_events.index)
    for event_type, weight in weights.items():
        if event_type in repo_events.columns:
            score += repo_events[event_type] * weight
    return score


repo_events["weighted_score"] = compute_weighted_score(repo_events, WEIGHTS)

weighted_ranking = repo_events["weighted_score"].nlargest(TOP_N)
print(f"Top {TOP_N} repos by weighted score:")
print(weighted_ranking.to_frame("score"))

Top 30 repos by weighted score:
               score
repo_id             
1103012935  126524.5
678894831    69287.6
4542716      51204.6
197275551    49789.8
1128494107   47228.9
1076470140   46987.7
1099385184   41945.2
1156956890   41912.9
1156607103   40120.1
1119742099   28324.1
1156518861   27315.5
1051430770   26806.4
1051442401   26238.8
1102369740   23052.4
970201570    22620.8
1026691215   21560.4
408894306    20701.5
171072967    20556.9
521028707    18989.8
52855516     18968.9
887068329    17090.8
1149691064   16545.6
1125932955   16484.6
573518657    15149.5
65600975     14809.6
746930940    14789.0
1164751568   14696.8
1148461431   14598.7
1086369682   14477.8
1148802571   14312.4


## 6. Repo 이름 매핑

repo_id만으로는 어떤 repo인지 알 수 없으니, 상위 repo들의 이름을 BigQuery에서 가져옵니다.

In [5]:
import os

from gharchive.client import create_client

KEY_PATH = os.environ.get("GCP_KEY_PATH", "gcp-key.json")
client = create_client(KEY_PATH)

# 두 랭킹의 상위 repo_id 합집합
top_star_ids = set(star_ranking.index)
top_weighted_ids = set(weighted_ranking.index)
all_top_ids = top_star_ids | top_weighted_ids

ids_str = ", ".join(str(rid) for rid in all_top_ids)

name_query = f"""
SELECT DISTINCT repo.id AS repo_id, repo.name AS repo_name
FROM `githubarchive.day.20260314`
WHERE repo.id IN ({ids_str})
"""

repo_names = client.query(name_query).to_dataframe()
name_map = dict(zip(repo_names["repo_id"], repo_names["repo_name"]))

print(f"Resolved {len(name_map)} repo names out of {len(all_top_ids)} IDs")

Resolved 55 repo names out of 58 IDs


## 7. 랭킹 비교

In [6]:
def build_ranking_df(
    ranking: pd.Series, name_map: dict[int, str], col_name: str
) -> pd.DataFrame:
    """Build a readable ranking DataFrame with repo names."""
    result = ranking.to_frame(col_name).reset_index()
    result["repo_name"] = result["repo_id"].map(name_map).fillna("(unknown)")
    result["rank"] = range(1, len(result) + 1)
    return result[["rank", "repo_name", "repo_id", col_name]]


df_star = build_ranking_df(star_ranking, name_map, "stars")
df_weighted = build_ranking_df(weighted_ranking, name_map, "score")

print("=" * 60)
print("Star 기준 Top-N")
print("=" * 60)
print(df_star.to_string(index=False))

print()
print("=" * 60)
print("가중 점수 기준 Top-N")
print("=" * 60)
print(df_weighted.to_string(index=False))

Star 기준 Top-N
 rank                                     repo_name    repo_id    stars
    1                             openclaw/openclaw 1103012935  32056.0
    2                    msitarzewski/agency-agents 1075372545   8670.0
    3                                 ruvnet/RuView  997737944   8396.0
    4                          koala73/worldmonitor 1130564872   8183.0
    5                              obra/superpowers 1073224795   7959.0
    6               affaan-m/everything-claude-code 1136590548   7706.0
    7                        zeroclaw-labs/zeroclaw 1156956890   7673.0
    8                         karpathy/autoresearch 1174820787   7397.0
    9                             anthropics/skills 1061953414   5998.0
   10         hesamsheikh/awesome-openclaw-usecases 1153093670   5938.0
   11             VoltAgent/awesome-openclaw-skills 1141704318   5662.0
   12                         paperclipai/paperclip 1170821064   5218.0
   13                               sipeed/picocla

In [7]:
# 두 랭킹 간 겹치는 repo 비교
star_set = set(df_star["repo_id"])
weighted_set = set(df_weighted["repo_id"])
overlap = star_set & weighted_set

print(f"Star Top-{TOP_N}에만 있는 repo: {len(star_set - weighted_set)}")
print(f"가중 Top-{TOP_N}에만 있는 repo: {len(weighted_set - star_set)}")
print(f"공통:                        {len(overlap)}")

# 가중 점수에만 있는 repo (star 기준과 다른 결과)
only_weighted = df_weighted[~df_weighted["repo_id"].isin(star_set)]
if len(only_weighted) > 0:
    print(f"\n가중 점수에서만 Top-{TOP_N}에 진입한 repo:")
    print(only_weighted[["rank", "repo_name", "score"]].to_string(index=False))

Star Top-30에만 있는 repo: 28
가중 Top-30에만 있는 repo: 28
공통:                        2

가중 점수에서만 Top-30에 진입한 repo:
 rank                                       repo_name    score
    2       leandrohstein/transporteservico-urbs-data  69287.6
    3                                   NixOS/nixpkgs  51204.6
    4                           microsoft/winget-pkgs  49789.8
    5                        merge-demo/mergequeue-st  47228.9
    6                            gaston1799/RobloxLua  46987.7
    7                escapingwork/teenagerspopulation  41945.2
    9                     merge-demo/mergequeue-bazel  40120.1
   10                        merge-demo/mergequeue-wf  28324.1
   11                                       (unknown)  27315.5
   12                       sidarthus89/EVE-Data-Site  26806.4
   13                   sidarthus89/EVE-Data-Site-Dev  26238.8
   14                            cosmocws/zelenza_app  23052.4
   15                               shaehabultra/test  22620.8
   16      